# Decision economics

A lender uses a model score to decide whether to approve each loan and to estimate what that
decision is worth. This requires values for both outcomes: the return from repayment and the
cost of default.

Because those outcomes are not symmetric and do not scale together, each loan has its own
break-even probability. This notebook applies that probability to the lending decision and
evaluates the result in currency rather than AUC.

What it does:

- Compares three decision rules: approve every loan, use one break-even threshold for the book,
  and use a per-loan threshold that varies with the interest rate.
- Evaluates each rule using the cash that the loans repaid rather than the margin assumed by the
  model.
- Prices each loan using two values estimated on the training vintages: the margin earned per
  point of interest rate and the share of principal lost on default. The values are checked
  against the database before use.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

from credit_risk.data import (
    load_loans,
    load_outcomes,
    DB_PATH
)
from credit_risk.split import out_of_time_split
from credit_risk.model import (
    build_lgbm,
    LC_VERDICT_NUMERIC, LC_VERDICT_CATEGORICAL,
    UNDERWRITER_NUMERIC, UNDERWRITER_CATEGORICAL,
)
from credit_risk.evaluate import (
    expected_profit,
    breakeven_probability,
    MARGIN_PER_RATE_POINT,
    LOSS_FRACTION,
)

TARGET = "target_bad"
NUMERIC = UNDERWRITER_NUMERIC + LC_VERDICT_NUMERIC
CATEGORICAL = UNDERWRITER_CATEGORICAL + LC_VERDICT_CATEGORICAL

df = load_loans()
train, val, _ = out_of_time_split(df)

cols = NUMERIC + CATEGORICAL
pipe = build_lgbm(NUMERIC, CATEGORICAL)
pipe.fit(train[cols], train[TARGET])

proba = pipe.predict_proba(val[cols])
val = val.assign(proba=proba[:, 1])

print(
    f"validation {len(val)} loans, mean predicted PD {val['proba'].mean():.3f}, "
    f"bad rate {val[TARGET].mean():.3f}"
)

validation 154703 loans, mean predicted PD 0.130, bad rate 0.150


## What each outcome is worth

`evaluate.py` stores the margin slope and loss fraction as defaults, allowing the rule to run
without a database. `sql/30_loan_economics.sql` estimates both values on the training vintages
from payment columns that become available only after origination. They are used here to estimate
business parameters; using them as model features would cause leakage.

The stored defaults can diverge from the database estimates over time. The cell below reads the
estimates from the database and displays both sets of values together.

In [2]:
statements = Path("../sql/30_loan_economics.sql").read_text().split(";")
economics_sql = [s for s in statements if s.strip()][-1]

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    estimated = con.execute(economics_sql).df().iloc[0]

pd.DataFrame({
    "from sql": {
        "margin_per_rate_point": estimated["margin_per_rate_point"],
        "loss_fraction": estimated["loss_fraction"],
    },
    "in evaluate.py": {
        "margin_per_rate_point": MARGIN_PER_RATE_POINT,
        "loss_fraction": LOSS_FRACTION,
    },
})

,from sql,in evaluate.py
margin_per_rate_point,0.0133,0.0133
loss_fraction,0.3543,0.3543


## Three policies

The notebook applies three rules to the same validation loans:

- **Approve all.** Fund every loan. This is the baseline against which the value of the other
  rules is measured.
- **Single break-even.** Approve a loan when its predicted default probability is below the
  break-even probability of the amount-weighted average rate. The threshold is derived from the
  economics rather than tuned on validation profit. The average rate is calculated on the
  training vintages, so the rule does not use the loans on which it is evaluated.
- **Expected profit.** Approve a loan when its expected profit is positive, which makes the
  threshold vary with the loan's rate.

Each rule is evaluated using each loan's realised cash flows rather than the margin slope used to
set the expected-profit threshold. The expected-profit rule is therefore evaluated against
observed cash flows rather than its own payoff assumptions.

In [3]:
outcomes = load_outcomes().drop(columns=["loan_amnt"])
scored = val.merge(outcomes, on="id")
assert len(scored) == len(val)

scored["realised_profit"] = (
    scored["total_rec_prncp"] + scored["recoveries"] + scored["total_rec_int"] - scored["loan_amnt"]
)

scored["exp_profit"] = expected_profit(scored["proba"], scored["loan_amnt"], scored["int_rate"])

r_bar = (train["int_rate"] * train["loan_amnt"]).sum() / train["loan_amnt"].sum()
c = breakeven_probability(r_bar)
print(f"single break-even: {c:.3f}, from an average rate of {r_bar:.1f}%")

policies = {
    "approve all": pd.Series(True, index=scored.index),
    "single break-even": scored["proba"] < c,
    "expected profit": scored["exp_profit"] > 0,
}

rows = {}
for name, approve in policies.items():
    taken = scored[approve]
    rows[name] = {
        "approved": len(taken),
        "total_profit": taken["realised_profit"].sum(),
        "profit_per_loan": taken["realised_profit"].mean(),
        "bad_rate": taken[TARGET].mean(),
    }

policy_results = pd.DataFrame(rows).T
policy_results

single break-even: 0.316, from an average rate of 12.3%


,approved,total_profit,profit_per_loan,bad_rate
approve all,154703.0,1.321967e+08,854.519481,0.150016
single break-even,151052.0,1.330806e+08,881.024863,0.144262
expected profit,154273.0,1.323583e+08,857.948790,0.149359


In [4]:
be = scored["proba"] < c
ep = scored["exp_profit"] > 0

print(f"single break-even rejects {(~be).sum()}, expected profit rejects {(~ep).sum()}")

extra = scored[ep & ~be]
print(
    f"kept only by expected profit: {len(extra)} loans at "
    f"{extra['int_rate'].mean():.0f}% average rate, "
    f"{extra['proba'].mean():.0%} mean predicted PD, "
    f"{extra[TARGET].mean():.0%} observed default, "
    f"{extra['realised_profit'].sum() / 1e6:.1f}M realised"
)

profit_check = pd.Series({
    "training payoff at predicted probabilities": expected_profit(
        extra["proba"], extra["loan_amnt"], extra["int_rate"]
    ).sum(),
    "training payoff at observed outcomes": expected_profit(
        extra[TARGET], extra["loan_amnt"], extra["int_rate"]
    ).sum(),
    "realised cash flows": extra["realised_profit"].sum(),
}, name="total_profit")
profit_check

single break-even rejects 3651, expected profit rejects 430
kept only by expected profit: 3296 loans at 19% average rate, 35% mean predicted PD, 39% observed default, -0.7M realised


training payoff at predicted probabilities    1.673081e+06
training payoff at observed outcomes          4.769206e+05
realised cash flows                          -7.355606e+05
Name: total_profit, dtype: float64

## Assumptions

The calculation does not discount future cash flows, so a euro received in month 36 is treated
like a euro received today. Prepayment is not modelled separately because the realised margin
already includes the lower interest earned when a loan is repaid early. The economic estimates
come from past loans and apply only while pricing and recovery behave as they did during that
period.

## Conclusions

The three policies produce between 132.2M and 133.1M on the validation book, a difference of
less than 1%. This portfolio contains only loans that Lending Club funded, and approving or
rejecting loans changes total profit little within that population.

The single break-even policy returns 133.1M, compared with 132.4M for per-loan pricing. The
difference is concentrated among loans on which the two rules disagree. The per-loan break-even
probability rises with the interest rate, so that rule retains high-rate loans that the single
threshold rejects. Their mean predicted PD is 35%, while their observed default rate is 39%.
Under the training-estimated payoff assumptions, the profit estimate falls from 1.7M with
predicted probabilities to 0.5M with observed outcomes. Their realised cash flows instead show a
loss of 0.7M. The first gap points to PD underprediction, while the second points to payoff
misspecification.

The per-loan rule depends on the predicted probabilities and on payoff assumptions estimated from
the training vintages. Errors in either input affect which high-rate loans it retains.

This result applies only to Lending Club's funded loans. Rejected applicants are absent, so the
approve-all result cannot be extended to the full applicant pool. The figures also assume no
discounting and that recovery behaviour remains as it was during the training period. Within this
scope, the model supplies the ranking and probability estimates used by the policies, but the
choice among these three rules changes total profit little.